# PDF Scan v2 - Section-first rebuild

This notebook implements **Phase A** from `PDF_SCAN_PIPELINE_IMPLEMENTATION_PLAN.md`.

- **Phase A**: config surface, run artifacts under `runs/{run_id}/`, structured logging, per-stage metrics, and a PDF manifest.
- Later phases will add parsing, canonical section construction, query planning, retrieval, reranking, and calibration.

**Ground rules for this version**

- Prefer loud, detailed errors over silent failure.
- Each major code cell ends with a compact QC summary.
- No retrieval or ranking happens yet in this notebook version.

## Step 0 - Edit your inputs

Edit the next cell, then run the notebook top-to-bottom.

What this Phase A notebook should produce:

- a stable `run_id`
- `runs/<run_id>/config.json`
- `runs/<run_id>/pdf_manifest.json`
- `runs/<run_id>/logs.jsonl`
- `runs/<run_id>/metrics.json`
- placeholder directories for later phases

In [ ]:
# -----------------------------
# USER INPUTS (edit this cell)
# -----------------------------

import re

CHAPTER_TITLE = "Technische Grundlagen: Zero Trust Architecture (ZTA) in Unternehmensnetzwerken"

CHAPTER_DESCRIPTION = """
Ziel ist eine prazise, technische Fundierung von Zero Trust Architecture (ZTA) fur Unternehmens-IT (On-Prem, Cloud, Hybrid),
um spater eine konkrete ZTA-Einfuhrung bewerten und planen zu konnen. Dazu gehoren Begriffsdefinition, Kernprinzipien,
Referenzarchitekturen, Telemetrie, kontinuierliche Bewertung, Migration in Legacy-Umgebungen und messbare Bewertungskriterien.
Produktvergleiche, Buyer's Guides und rein allgemeine Kryptographie-Einfuhrungen gehoren nicht in den Scope.
""".strip()

# Option A: explicit PDF list
PDF_SOURCES = [
    # {"label": "paper_1", "path": r"C:\\path\\to\\paper.pdf"},
]

# Option B: discover PDFs from a directory when PDF_SOURCES is empty
PDF_DIR = r""
PDF_GLOB = "*.pdf"
PDF_RECURSIVE = False
MAX_PDFS = 20

PIPELINE_VERSION = "pdf_scan_v2"
FORCE_REBUILD_PHASE_A = False


def _fmt_int(x) -> str:
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def _truncate(text: str, max_len: int = 120) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "...")


def print_section(title: str, width: int = 80, char: str = "=") -> None:
    line = char * width
    print(line)
    print(title)
    print(line)


def print_kv(d: dict, key_width: int = 26) -> None:
    for k, v in d.items():
        print(f"{str(k):<{key_width}} {v}")


if not str(CHAPTER_TITLE or "").strip():
    raise ValueError("CHAPTER_TITLE must not be empty.")
if not str(CHAPTER_DESCRIPTION or "").strip():
    raise ValueError("CHAPTER_DESCRIPTION must not be empty.")

desc_words = len(re.findall(r"\w+", CHAPTER_DESCRIPTION, flags=re.UNICODE))
source_mode = "PDF_SOURCES" if PDF_SOURCES else ("PDF_DIR" if str(PDF_DIR or "").strip() else "unset")

print_section("User Inputs")
print_kv(
    {
        "chapter_title": _truncate(CHAPTER_TITLE, 90),
        "chapter_desc_chars": _fmt_int(len(CHAPTER_DESCRIPTION)),
        "chapter_desc_words": _fmt_int(desc_words),
        "pdf_source_mode": source_mode,
        "pipeline_version": PIPELINE_VERSION,
        "force_rebuild_phase_a": FORCE_REBUILD_PHASE_A,
    }
)

print_section("User Inputs - PDF Discovery Config")
print_kv(
    {
        "pdf_sources_count": _fmt_int(len(PDF_SOURCES)),
        "pdf_dir": PDF_DIR or "<empty>",
        "pdf_glob": PDF_GLOB,
        "pdf_recursive": PDF_RECURSIVE,
        "max_pdfs": MAX_PDFS,
    }
)

---
# Phase A - Config, env loading, run artifacts, and structured logging
---

In [ ]:
# Phase A.0 - Imports, repo resolution, env loading, and helper functions

import json
import logging
import os
import hashlib
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None


def _find_repo_root_and_notebook_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for base in candidates:
        pdf_scan_dir = base / "pdf-scan"
        if pdf_scan_dir.exists() and pdf_scan_dir.is_dir():
            return base, pdf_scan_dir
        if base.name == "pdf-scan":
            return base.parent, base
    raise RuntimeError("Could not resolve repo root / pdf-scan directory from current working directory.")


REPO_ROOT, NOTEBOOK_DIR = _find_repo_root_and_notebook_dir()

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

OPENAI_API_KEY = (os.getenv("OPENAI_API_KEY") or "").strip()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def fmt_float(x: Any, nd: int = 3) -> str:
    try:
        return f"{float(x):.{int(nd)}f}"
    except Exception:
        return str(x)


def fmt_ms(ms: Any) -> str:
    try:
        val = float(ms)
    except Exception:
        return str(ms)
    if val < 1000:
        return f"{val:.0f}ms"
    return f"{(val / 1000.0):.2f}s"


def print_table(rows: List[Dict[str, Any]], *, columns: List[str], max_rows: int = 50, max_col_width: int = 60) -> None:
    rows = list(rows or [])
    if not rows:
        print("<empty>")
        return

    show = rows[: int(max_rows)]
    cols = list(columns)

    def cell(row: Dict[str, Any], col: str) -> str:
        value = row.get(col, "")
        if value is None:
            value = ""
        text = str(value).replace("\r", " ").replace("\n", " ")
        return text if len(text) <= max_col_width else (text[: max_col_width - 3] + "...")

    widths = {}
    for col in cols:
        widths[col] = min(
            max(len(col), max(len(cell(r, col)) for r in show)),
            max_col_width,
        )

    header = " | ".join(f"{col:<{widths[col]}}" for col in cols)
    sep = "-+-".join("-" * widths[col] for col in cols)
    print(header)
    print(sep)
    for row in show:
        print(" | ".join(f"{cell(row, col):<{widths[col]}}" for col in cols))
    if len(rows) > len(show):
        print(f"... (+{len(rows) - len(show)} more rows)")


def qc_row(check: str, status: str, value: Any, expected: str, why: str, fix: str) -> Dict[str, Any]:
    return {
        "check": str(check),
        "status": str(status),
        "value": str(value),
        "expected": str(expected),
        "why": str(why),
        "fix": str(fix),
    }


def stable_hash(*parts: str, length: int = 24) -> str:
    payload = "\n".join([(p or "").strip().replace("\r\n", "\n") for p in parts])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[: int(length)]


def _json_default(obj: Any):
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def write_json(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default) + "\n", encoding="utf-8")
    tmp.replace(path)


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def append_jsonl(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=_json_default) + "\n")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def inspect_pdf(path: Path) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "page_count": None,
        "has_outline": None,
        "inspect_status": "not_attempted",
    }
    if fitz is None:
        out["inspect_status"] = "fitz_unavailable"
        return out
    try:
        with fitz.open(path) as doc:
            out["page_count"] = int(doc.page_count)
            try:
                out["has_outline"] = bool(doc.get_toc())
            except Exception:
                out["has_outline"] = None
        out["inspect_status"] = "ok"
    except Exception as e:
        out["inspect_status"] = f"error:{type(e).__name__}"
    return out


print_section("Phase A.0 - Environment")
print_kv(
    {
        "repo_root": REPO_ROOT,
        "notebook_dir": NOTEBOOK_DIR,
        "openai_api_key_present": bool(OPENAI_API_KEY),
        "pymupdf_available": bool(fitz is not None),
        "utc_now": utc_now_iso(),
    }
)

In [ ]:
# Phase A.1 - Config models, PDF source resolution, run artifacts, and logging helpers

@dataclass
class PdfSource:
    label: str
    path: Path


@dataclass
class RunArtifacts:
    config_json: Path
    pdf_manifest_json: Path
    query_plan_json: Path
    parser_dir: Path
    normalized_dir: Path
    retrieval_dir: Path
    rerank_dir: Path
    final_dir: Path
    logs_jsonl: Path
    run_log: Path
    metrics_json: Path

    @classmethod
    def from_run_dir(cls, run_dir: Path) -> "RunArtifacts":
        return cls(
            config_json=run_dir / "config.json",
            pdf_manifest_json=run_dir / "pdf_manifest.json",
            query_plan_json=run_dir / "query_plan.json",
            parser_dir=run_dir / "parser",
            normalized_dir=run_dir / "normalized",
            retrieval_dir=run_dir / "retrieval",
            rerank_dir=run_dir / "rerank",
            final_dir=run_dir / "final",
            logs_jsonl=run_dir / "logs.jsonl",
            run_log=run_dir / "run.log",
            metrics_json=run_dir / "metrics.json",
        )


@dataclass
class PipelineConfig:
    pipeline_version: str
    chapter_title: str
    chapter_spec_text: str
    runs_root: Path
    openai_api_key_present: bool
    force_rebuild_phase_a: bool
    pdf_sources: List[PdfSource]
    pdf_dir_raw: str
    pdf_glob: str
    pdf_recursive: bool
    max_pdfs: int

    def to_snapshot(self) -> Dict[str, Any]:
        return {
            "pipeline_version": self.pipeline_version,
            "chapter_title": self.chapter_title,
            "chapter_spec_text_chars": len(self.chapter_spec_text),
            "runs_root": self.runs_root,
            "openai_api_key_present": self.openai_api_key_present,
            "force_rebuild_phase_a": self.force_rebuild_phase_a,
            "pdf_sources": [{"label": s.label, "path": str(s.path)} for s in self.pdf_sources],
            "pdf_dir_raw": self.pdf_dir_raw,
            "pdf_glob": self.pdf_glob,
            "pdf_recursive": self.pdf_recursive,
            "max_pdfs": self.max_pdfs,
        }


@dataclass
class RunContext:
    repo_root: Path
    notebook_dir: Path
    run_id: str
    run_dir: Path
    artifacts: RunArtifacts

    def create_artifact_skeleton(self, overwrite: bool = False) -> None:
        ensure_dir(self.run_dir)
        ensure_dir(self.artifacts.parser_dir)
        ensure_dir(self.artifacts.normalized_dir)
        ensure_dir(self.artifacts.retrieval_dir)
        ensure_dir(self.artifacts.rerank_dir)
        ensure_dir(self.artifacts.final_dir)

        placeholders: Dict[Path, Any] = {
            self.artifacts.query_plan_json: {"status": "not_run", "phase": "query_planner"},
            self.artifacts.metrics_json: {"run_id": self.run_id, "stages": {}},
        }
        for path, payload in placeholders.items():
            if overwrite or (not path.exists()):
                write_json(path, payload)

        for path in [self.artifacts.logs_jsonl, self.artifacts.run_log]:
            if overwrite or (not path.exists()):
                ensure_dir(path.parent)
                path.write_text("", encoding="utf-8")

        for rel in [
            self.artifacts.normalized_dir / "documents.jsonl",
            self.artifacts.normalized_dir / "sections.jsonl",
            self.artifacts.normalized_dir / "passages.jsonl",
            self.artifacts.retrieval_dir / "fused_candidates.jsonl",
            self.artifacts.rerank_dir / "cross_encoder.jsonl",
            self.artifacts.final_dir / "output.json",
        ]:
            if overwrite or (not rel.exists()):
                ensure_dir(rel.parent)
                if rel.suffix == ".json":
                    write_json(rel, {"status": "not_run"})
                else:
                    rel.write_text("", encoding="utf-8")


def _resolve_existing_path(raw: str, *, expect_dir: bool) -> Path:
    p = Path(raw).expanduser()
    candidates = [p]
    if not p.is_absolute():
        candidates.extend([NOTEBOOK_DIR / p, REPO_ROOT / p, Path.cwd().resolve() / p])
    seen: List[Path] = []
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.append(cand)
        if cand.exists() and ((cand.is_dir() and expect_dir) or (cand.is_file() and not expect_dir)):
            return cand
    return candidates[0].resolve()


def _normalize_pdf_sources(raw_sources: List[Dict[str, Any]]) -> List[PdfSource]:
    out: List[PdfSource] = []
    seen_labels: Dict[str, int] = {}
    for item in raw_sources or []:
        if not isinstance(item, dict):
            continue
        path_raw = str(item.get("path") or "").strip()
        if not path_raw:
            continue
        path = _resolve_existing_path(path_raw, expect_dir=False)
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {path}")
        label = str(item.get("label") or path.stem).strip() or path.stem
        n = seen_labels.get(label, 0) + 1
        seen_labels[label] = n
        if n > 1:
            label = f"{label} ({n})"
        out.append(PdfSource(label=label, path=path))
    return out


def resolve_pdf_sources() -> List[PdfSource]:
    explicit = _normalize_pdf_sources(PDF_SOURCES)
    if explicit:
        return explicit[: int(MAX_PDFS)]

    pdf_dir = str(PDF_DIR or "").strip()
    if not pdf_dir:
        raise RuntimeError("No PDFs configured. Set PDF_SOURCES or PDF_DIR.")

    root = _resolve_existing_path(pdf_dir, expect_dir=True)
    if not root.exists():
        raise FileNotFoundError(f"PDF_DIR not found: {root}")

    paths = sorted(root.rglob(PDF_GLOB) if bool(PDF_RECURSIVE) else root.glob(PDF_GLOB))
    paths = [p.resolve() for p in paths if p.is_file()]
    if not paths:
        raise FileNotFoundError(f"No PDFs found in {root} with pattern {PDF_GLOB!r}")

    out: List[PdfSource] = []
    seen_labels: Dict[str, int] = {}
    for path in paths[: int(MAX_PDFS)]:
        label = path.stem
        n = seen_labels.get(label, 0) + 1
        seen_labels[label] = n
        if n > 1:
            label = f"{label} ({n})"
        out.append(PdfSource(label=label, path=path))
    return out


def compute_run_id(chapter_title: str, chapter_spec_text: str, pipeline_version: str, manifest_rows: List[Dict[str, Any]]) -> str:
    doc_parts = [f"{row.get('label')}::{row.get('sha256')}" for row in manifest_rows]
    return stable_hash(pipeline_version, chapter_title, chapter_spec_text, "\n".join(doc_parts), length=24)


def load_metrics(run_ctx: RunContext) -> Dict[str, Any]:
    if run_ctx.artifacts.metrics_json.exists():
        try:
            return read_json(run_ctx.artifacts.metrics_json)
        except Exception:
            return {"run_id": run_ctx.run_id, "stages": {}}
    return {"run_id": run_ctx.run_id, "stages": {}}


def save_metrics(run_ctx: RunContext, metrics: Dict[str, Any]) -> None:
    write_json(run_ctx.artifacts.metrics_json, metrics)


def setup_run_logger(run_ctx: RunContext) -> logging.Logger:
    logger_name = f"pdf_scan_v2.{run_ctx.run_id}"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    logger.handlers = []
    logger.propagate = False

    fh = logging.FileHandler(run_ctx.artifacts.run_log, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    return logger


def log_event(run_ctx: RunContext, *, stage: str, event: str, **payload: Any) -> None:
    append_jsonl(
        run_ctx.artifacts.logs_jsonl,
        {
            "ts_utc": utc_now_iso(),
            "stage": stage,
            "event": event,
            **payload,
        },
    )


@contextmanager
def stage_timer(run_ctx: RunContext, stage: str):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)
        metrics = load_metrics(run_ctx)
        metrics.setdefault("stages", {}).setdefault(stage, {})["elapsed_ms"] = elapsed_ms
        metrics["stages"][stage]["finished_at_utc"] = utc_now_iso()
        save_metrics(run_ctx, metrics)
        log_event(run_ctx, stage=stage, event="stage_finished", elapsed_ms=elapsed_ms)

In [ ]:
# Phase A.2 - Resolve PDFs, create run context, write artifacts, and print QC summary

resolved_sources = resolve_pdf_sources()
if not resolved_sources:
    raise RuntimeError("resolve_pdf_sources() returned no PDFs.")

pdf_manifest_rows: List[Dict[str, Any]] = []
for src in resolved_sources:
    stat = src.path.stat()
    inspect = inspect_pdf(src.path)
    pdf_manifest_rows.append(
        {
            "label": src.label,
            "path": str(src.path),
            "file_name": src.path.name,
            "size_mb": round(float(stat.st_size) / (1024.0 * 1024.0), 3),
            "sha256": sha256_file(src.path),
            "page_count": inspect.get("page_count"),
            "has_outline": inspect.get("has_outline"),
            "inspect_status": inspect.get("inspect_status"),
            "mtime_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).replace(microsecond=0).isoformat(),
        }
    )

run_id = compute_run_id(CHAPTER_TITLE, CHAPTER_DESCRIPTION, PIPELINE_VERSION, pdf_manifest_rows)
runs_root = ensure_dir((NOTEBOOK_DIR / "runs").resolve())
run_dir = runs_root / run_id
artifacts = RunArtifacts.from_run_dir(run_dir)
run_ctx = RunContext(repo_root=REPO_ROOT, notebook_dir=NOTEBOOK_DIR, run_id=run_id, run_dir=run_dir, artifacts=artifacts)

cfg = PipelineConfig(
    pipeline_version=PIPELINE_VERSION,
    chapter_title=CHAPTER_TITLE,
    chapter_spec_text=CHAPTER_DESCRIPTION,
    runs_root=runs_root,
    openai_api_key_present=bool(OPENAI_API_KEY),
    force_rebuild_phase_a=bool(FORCE_REBUILD_PHASE_A),
    pdf_sources=resolved_sources,
    pdf_dir_raw=str(PDF_DIR or ""),
    pdf_glob=str(PDF_GLOB or "*.pdf"),
    pdf_recursive=bool(PDF_RECURSIVE),
    max_pdfs=int(MAX_PDFS),
)

with stage_timer(run_ctx, "phase_a"):
    run_ctx.create_artifact_skeleton(overwrite=bool(FORCE_REBUILD_PHASE_A))
    logger = setup_run_logger(run_ctx)
    logger.info("Phase A initialized | run_id=%s | run_dir=%s", run_ctx.run_id, run_ctx.run_dir)

    write_json(run_ctx.artifacts.config_json, cfg.to_snapshot())
    write_json(
        run_ctx.artifacts.pdf_manifest_json,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "pdf_count": len(pdf_manifest_rows),
            "pdfs": pdf_manifest_rows,
        },
    )

    metrics = load_metrics(run_ctx)
    metrics.setdefault("stages", {}).setdefault("phase_a", {}).update(
        {
            "initialized_at_utc": utc_now_iso(),
            "pdf_count": len(pdf_manifest_rows),
            "has_openai_api_key": bool(OPENAI_API_KEY),
            "pymupdf_available": bool(fitz is not None),
        }
    )
    save_metrics(run_ctx, metrics)

    log_event(
        run_ctx,
        stage="phase_a",
        event="run_initialized",
        run_id=run_ctx.run_id,
        run_dir=str(run_ctx.run_dir),
        pdf_count=len(pdf_manifest_rows),
        has_openai_api_key=bool(OPENAI_API_KEY),
    )

expected_paths = [
    run_ctx.artifacts.config_json,
    run_ctx.artifacts.pdf_manifest_json,
    run_ctx.artifacts.query_plan_json,
    run_ctx.artifacts.parser_dir,
    run_ctx.artifacts.normalized_dir,
    run_ctx.artifacts.retrieval_dir,
    run_ctx.artifacts.rerank_dir,
    run_ctx.artifacts.final_dir,
    run_ctx.artifacts.logs_jsonl,
    run_ctx.artifacts.run_log,
    run_ctx.artifacts.metrics_json,
]
missing_paths = [str(p) for p in expected_paths if not p.exists()]

artifact_rows = [
    {"artifact": "config_json", "path": run_ctx.artifacts.config_json, "exists": run_ctx.artifacts.config_json.exists()},
    {"artifact": "pdf_manifest_json", "path": run_ctx.artifacts.pdf_manifest_json, "exists": run_ctx.artifacts.pdf_manifest_json.exists()},
    {"artifact": "query_plan_json", "path": run_ctx.artifacts.query_plan_json, "exists": run_ctx.artifacts.query_plan_json.exists()},
    {"artifact": "parser_dir", "path": run_ctx.artifacts.parser_dir, "exists": run_ctx.artifacts.parser_dir.exists()},
    {"artifact": "normalized_dir", "path": run_ctx.artifacts.normalized_dir, "exists": run_ctx.artifacts.normalized_dir.exists()},
    {"artifact": "retrieval_dir", "path": run_ctx.artifacts.retrieval_dir, "exists": run_ctx.artifacts.retrieval_dir.exists()},
    {"artifact": "rerank_dir", "path": run_ctx.artifacts.rerank_dir, "exists": run_ctx.artifacts.rerank_dir.exists()},
    {"artifact": "final_dir", "path": run_ctx.artifacts.final_dir, "exists": run_ctx.artifacts.final_dir.exists()},
    {"artifact": "logs_jsonl", "path": run_ctx.artifacts.logs_jsonl, "exists": run_ctx.artifacts.logs_jsonl.exists()},
    {"artifact": "metrics_json", "path": run_ctx.artifacts.metrics_json, "exists": run_ctx.artifacts.metrics_json.exists()},
]

qc_rows = []
qc_rows.append(
    qc_row(
        check="artifact_skeleton",
        status="OK" if not missing_paths else "FAIL",
        value="all present" if not missing_paths else ("missing: " + ", ".join(missing_paths[:4])),
        expected="all expected artifact paths exist",
        why="later phases rely on deterministic artifact locations",
        fix="re-run Phase A or inspect permissions / path resolution",
    )
)
qc_rows.append(
    qc_row(
        check="openai_api_key",
        status="OK" if bool(OPENAI_API_KEY) else "WARN",
        value=bool(OPENAI_API_KEY),
        expected="True before query-planning and LLM phases",
        why="later phases use the OpenAI API",
        fix="set OPENAI_API_KEY in .env before Phase D",
    )
)
qc_rows.append(
    qc_row(
        check="pymupdf_available",
        status="OK" if bool(fitz is not None) else "WARN",
        value=bool(fitz is not None),
        expected="True",
        why="PyMuPDF is used for PDF inspection and later fallback parsing",
        fix="install PyMuPDF before Phase B if this is False",
    )
)
qc_rows.append(
    qc_row(
        check="pdf_count",
        status="OK" if len(pdf_manifest_rows) >= 1 else "FAIL",
        value=len(pdf_manifest_rows),
        expected=">= 1",
        why="the pipeline needs at least one PDF input",
        fix="add PDFs to PDF_SOURCES or PDF_DIR",
    )
)

RUN_CONTEXT = run_ctx
CONFIG = cfg
PDF_MANIFEST = pdf_manifest_rows

print_section("Phase A.2 - Run Context")
print_kv(
    {
        "run_id": run_ctx.run_id,
        "run_dir": run_ctx.run_dir,
        "pdf_count": len(pdf_manifest_rows),
        "chapter_title": _truncate(CHAPTER_TITLE, 90),
        "pipeline_version": PIPELINE_VERSION,
    }
)

print_section("Phase A.2 - PDF Manifest Preview")
print_table(
    pdf_manifest_rows,
    columns=["label", "file_name", "page_count", "has_outline", "size_mb", "inspect_status"],
    max_rows=20,
)

print_section("Phase A.2 - Artifact Preview")
print_table(artifact_rows, columns=["artifact", "exists", "path"], max_rows=20, max_col_width=70)

print_section("Phase A.2 - QC")
print_table(qc_rows, columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)